# From Pandas to SQLite: Working with Databases

This notebook introduces SQLite databases and shows how pandas can serve as a bridge between Python and SQL. SQLite is a lightweight database system that stores everything in a single file, requiring no separate server installation. It's built into Python, making it ideal for learning database concepts and for projects that don't require a full database server.

By the end of this notebook, you will understand how to create and connect to SQLite databases, create tables with appropriate data types, insert data, query data with filtering conditions, delete records, and move data between pandas DataFrames and database tables.

## Why Databases?

You might wonder why we need databases when CSV and Excel files work well for storing data. Databases offer several advantages:

**Structured queries:** SQL provides a standardised way to ask questions about your data. Instead of writing Python code to filter a DataFrame, you can write a query that describes what you want.

**Data integrity:** Databases can enforce rules about what data is allowed, preventing errors before they happen.

**Multiple tables:** Databases naturally handle related tables. A school database might have separate tables for students, courses, and enrolments, all linked together.

**Persistence:** The database file exists independently of your Python session. You can close your notebook, reopen it later, and the data is still there.

For this course, we use SQLite because it requires no setup beyond importing Python's built-in sqlite3 module.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Environment detection for Colab compatibility
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running in standard Jupyter environment")

## Creating and Connecting to a Database

SQLite stores an entire database in a single file. When you connect to a database file that doesn't exist, SQLite creates it for you.

In [ ]:
# Create a connection to a database file
# If the file doesn't exist, SQLite creates it
conn = sqlite3.connect('college.db')

print("Connected to college.db")
print(f"Connection object: {conn}")

The connection object (`conn`) is your link to the database. You'll use it for all database operations. When you're finished working with the database, you should close the connection to release the file.

In [ ]:
# The cursor object executes SQL commands
cursor = conn.cursor()

print("Cursor created - ready to execute SQL")

## SQL Data Types

When creating tables, you specify what type of data each column can hold. SQLite uses these main data types:

| Type | Description | Examples |
|------|-------------|----------|
| TEXT | Text strings | 'Dublin', 'Computing', 'Dr. Smith' |
| INTEGER | Whole numbers | 42, -7, 2024 |
| REAL | Decimal numbers | 3.14, 98.6, -0.5 |
| BLOB | Binary data | Images, files (rarely used directly) |
| NULL | Missing/unknown | Represents absence of data |

Choosing the right type helps ensure data quality. If a column should only contain numbers, declaring it as INTEGER prevents accidentally storing text there.

## Creating Tables

Tables are created using the `CREATE TABLE` statement. You specify the table name and define each column with its name and data type.

In [ ]:
# Create a students table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        name TEXT,
        age INTEGER,
        programme TEXT
    )
''')

print("Created students table")

The `IF NOT EXISTS` clause is helpful during development. Without it, trying to create a table that already exists would cause an error. With this clause, SQLite simply skips the creation if the table is already there.

In [ ]:
# Create a modules table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS modules (
        module_name TEXT,
        hours_per_week INTEGER,
        lecturer TEXT
    )
''')

print("Created modules table")

In [ ]:
# Check what tables exist in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("Tables in database:")
for table in tables:
    print(f"  - {table[0]}")

## Inserting Data

The `INSERT INTO` statement adds rows to a table. You specify which columns you're filling and provide the values.

In [ ]:
# Insert a single row
cursor.execute('''
    INSERT INTO students (name, age, programme)
    VALUES ('Maya Chen', 19, 'Computing')
''')

print("Inserted one student")

In [ ]:
# Insert multiple rows at once
cursor.execute('''
    INSERT INTO students (name, age, programme)
    VALUES 
        ('Fatima Al-Rahman', 22, 'Business'),
        ('Kofi Mensah', 20, 'Design'),
        ('Aoife Kelly', 21, 'Computing'),
        ('Liam Murphy', 19, 'Business')
''')

print("Inserted four more students")

In [ ]:
# Insert data into modules table
cursor.execute('''
    INSERT INTO modules (module_name, hours_per_week, lecturer)
    VALUES 
        ('Database Methods', 4, 'Dr. Walsh'),
        ('Programming Fundamentals', 5, 'Ms. Patel'),
        ('Web Development', 6, 'Mr. O Sullivan'),
        ('Communications', 3, 'Dr. Ryan')
''')

print("Inserted modules")

In [ ]:
# Commit the changes to save them to the file
conn.commit()

print("Changes committed to database")

The `commit()` call is important. SQLite uses transactions, which means changes aren't permanently saved until you commit them. This allows you to make multiple changes and then either save them all together or undo them if something goes wrong.

## Querying Data with SELECT

The `SELECT` statement retrieves data from a table. The simplest form retrieves all columns and all rows.

In [ ]:
# Select all data from students
cursor.execute('SELECT * FROM students')
results = cursor.fetchall()

print("All students:")
for row in results:
    print(f"  {row}")

The asterisk (*) means "all columns". You can also select specific columns by name.

In [ ]:
# Select specific columns
cursor.execute('SELECT name, programme FROM students')
results = cursor.fetchall()

print("Student names and programmes:")
for row in results:
    print(f"  {row[0]} - {row[1]}")

### Filtering with WHERE

The `WHERE` clause filters rows based on conditions. Only rows that satisfy the condition are returned.

In [ ]:
# Filter: students aged 20 or older
cursor.execute('SELECT * FROM students WHERE age >= 20')
results = cursor.fetchall()

print("Students aged 20 or older:")
for row in results:
    print(f"  {row}")

In [ ]:
# Filter: students in Computing programme
cursor.execute("SELECT * FROM students WHERE programme = 'Computing'")
results = cursor.fetchall()

print("Computing students:")
for row in results:
    print(f"  {row}")

In [ ]:
# Filter: modules with 5 or more hours per week
cursor.execute('SELECT * FROM modules WHERE hours_per_week >= 5')
results = cursor.fetchall()

print("Modules with 5+ hours per week:")
for row in results:
    print(f"  {row}")

### Common WHERE Operators

| Operator | Meaning | Example |
|----------|---------|--------|
| = | Equal to | `age = 20` |
| != or <> | Not equal to | `programme != 'Business'` |
| > | Greater than | `hours_per_week > 4` |
| >= | Greater than or equal | `age >= 21` |
| < | Less than | `age < 20` |
| <= | Less than or equal | `hours_per_week <= 3` |
| LIKE | Pattern matching | `name LIKE 'A%'` (starts with A) |

In [ ]:
# LIKE with wildcards
# % matches any sequence of characters
cursor.execute("SELECT * FROM students WHERE name LIKE 'A%'")
results = cursor.fetchall()

print("Students whose name starts with A:")
for row in results:
    print(f"  {row}")

In [ ]:
# LIKE for contains
cursor.execute("SELECT * FROM modules WHERE lecturer LIKE '%Dr%'")
results = cursor.fetchall()

print("Modules taught by doctors:")
for row in results:
    print(f"  {row}")

## Deleting Data

The `DELETE FROM` statement removes rows from a table. Always use a `WHERE` clause to specify which rows to delete, otherwise all rows will be removed.

In [ ]:
# Check students before deletion
cursor.execute('SELECT * FROM students')
print("Before deletion:")
for row in cursor.fetchall():
    print(f"  {row}")

In [ ]:
# Delete a specific student
cursor.execute("DELETE FROM students WHERE name = 'Maya Chen'")
conn.commit()

print("Deleted Maya Chen")

In [ ]:
# Check students after deletion
cursor.execute('SELECT * FROM students')
print("After deletion:")
for row in cursor.fetchall():
    print(f"  {row}")

## The Pandas Bridge: read_sql() and to_sql()

Pandas provides convenient functions to move data between DataFrames and databases. This lets you use whichever tool is more appropriate for each task.

### Reading from Database to DataFrame

In [ ]:
# Read entire table into a DataFrame
students_df = pd.read_sql('SELECT * FROM students', conn)
students_df

In [ ]:
# Read with a filter - the filtering happens in SQL
computing_df = pd.read_sql("SELECT * FROM students WHERE programme = 'Computing'", conn)
computing_df

In [ ]:
# Read modules
modules_df = pd.read_sql('SELECT * FROM modules', conn)
modules_df

Once data is in a DataFrame, you can use all the pandas techniques you've learned: filtering, calculating new columns, visualisation, and so on.

In [ ]:
# Work with the DataFrame using pandas
print(f"Average age: {students_df['age'].mean():.1f}")
print(f"\nProgrammes: {students_df['programme'].unique()}")
print(f"\nStudents per programme:")
print(students_df['programme'].value_counts())

### Writing from DataFrame to Database

In [ ]:
# Create a new DataFrame
new_students = pd.DataFrame({
    'name': ['James Wong', 'Sofia Garcia', 'Erik Larsson'],
    'age': [23, 20, 22],
    'programme': ['Design', 'Computing', 'Business']
})

new_students

In [ ]:
# Write DataFrame to database
# if_exists options: 'fail', 'replace', 'append'
new_students.to_sql('students', conn, if_exists='append', index=False)

print("Added new students to database")

The `if_exists` parameter controls what happens if the table already exists:
- `'fail'`: Raise an error (default)
- `'replace'`: Drop the existing table and create a new one
- `'append'`: Add the new rows to the existing table

The `index=False` parameter prevents pandas from writing the DataFrame's index as a column in the database.

In [ ]:
# Verify the addition
all_students = pd.read_sql('SELECT * FROM students', conn)
print(f"Total students now: {len(all_students)}")
all_students

### Creating a New Table from a DataFrame

In [ ]:
# Create a DataFrame with lecturers
lecturers = pd.DataFrame({
    'name': ['Dr. Walsh', 'Ms. Patel', 'Mr. O Sullivan', 'Dr. Ryan'],
    'department': ['Computing', 'Computing', 'Computing', 'Business'],
    'years_experience': [15, 8, 12, 20]
})

# Write to a new table
lecturers.to_sql('lecturers', conn, if_exists='replace', index=False)

print("Created lecturers table")

In [ ]:
# Check tables in database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
for t in tables['name']:
    print(f"  - {t}")

## Putting It Together: A Complete Workflow

Let's walk through a complete example that demonstrates the typical workflow of working with databases and pandas together.

In [ ]:
# Create a fresh database for this example
demo_conn = sqlite3.connect('bookshop.db')

# Create a books DataFrame
books = pd.DataFrame({
    'title': ['The Great Gatsby', '1984', 'Pride and Prejudice', 'To Kill a Mockingbird', 'The Catcher in the Rye'],
    'author': ['F. Scott Fitzgerald', 'George Orwell', 'Jane Austen', 'Harper Lee', 'J.D. Salinger'],
    'year': [1925, 1949, 1813, 1960, 1951],
    'price': [12.99, 9.99, 8.99, 14.99, 11.99],
    'in_stock': [True, True, False, True, True]
})

# Save to database
books.to_sql('books', demo_conn, if_exists='replace', index=False)

print("Created bookshop database with books table")
books

In [ ]:
# Query: Books priced under 12 pounds
affordable = pd.read_sql('SELECT * FROM books WHERE price < 12', demo_conn)
print("Affordable books (under 12):")
affordable

In [ ]:
# Query: Books published before 1950
classics = pd.read_sql('SELECT title, author, year FROM books WHERE year < 1950', demo_conn)
print("Pre-1950 classics:")
classics

In [ ]:
# Add new books using SQL directly
cursor = demo_conn.cursor()
cursor.execute('''
    INSERT INTO books (title, author, year, price, in_stock)
    VALUES 
        ('Brave New World', 'Aldous Huxley', 1932, 10.99, 1),
        ('The Hobbit', 'J.R.R. Tolkien', 1937, 13.99, 1)
''')
demo_conn.commit()

print("Added two more books")

In [ ]:
# Read updated table
all_books = pd.read_sql('SELECT * FROM books', demo_conn)
print(f"Total books: {len(all_books)}")
all_books

In [ ]:
# Delete a book
cursor.execute("DELETE FROM books WHERE title = 'The Catcher in the Rye'")
demo_conn.commit()

# Verify
remaining = pd.read_sql('SELECT title FROM books', demo_conn)
print("Remaining books:")
for title in remaining['title']:
    print(f"  - {title}")

In [ ]:
# Close the demo connection
demo_conn.close()
print("Closed bookshop database connection")

## Closing Connections

When you're finished with a database, close the connection to release the file. This is especially important if other programs might need to access the database.

In [ ]:
# Close our main connection
conn.close()
print("Closed college.db connection")

You can also use Python's `with` statement to automatically close connections:

In [ ]:
# Using 'with' for automatic cleanup
with sqlite3.connect('college.db') as conn:
    df = pd.read_sql('SELECT * FROM students', conn)
    print(f"Read {len(df)} students")

# Connection is automatically closed when the 'with' block ends
print("Connection closed automatically")

## Practice Exercises

**Exercise 1:** Create a new database called `music.db`. Create a table called `songs` with columns for title (TEXT), artist (TEXT), year (INTEGER), and duration_seconds (INTEGER). Insert at least 4 songs.

In [ ]:
# Your code here


**Exercise 2:** Write a query to select all songs from your database that are longer than 3 minutes (180 seconds). Display the results.

In [ ]:
# Your code here


**Exercise 3:** Create a DataFrame with information about 3 albums (title, artist, year, num_tracks). Use `to_sql()` to save it to a new table called `albums` in your music database.

In [ ]:
# Your code here


**Exercise 4:** Delete one song from your songs table, then use `read_sql()` to load the remaining songs into a DataFrame and display them.

In [ ]:
# Your code here


## Summary

This notebook covered the fundamentals of working with SQLite databases:

**Connecting:** Use `sqlite3.connect('filename.db')` to create or open a database.

**Creating tables:** Use `CREATE TABLE` with column names and data types (TEXT, INTEGER, REAL).

**Inserting data:** Use `INSERT INTO` with column names and values.

**Querying:** Use `SELECT` to retrieve data, with `WHERE` to filter rows.

**Deleting:** Use `DELETE FROM` with a `WHERE` clause to remove specific rows.

**Pandas bridge:** Use `pd.read_sql()` to load database data into DataFrames, and `df.to_sql()` to write DataFrames to database tables.

**Cleanup:** Always close connections with `conn.close()` or use the `with` statement.

## Bibliography

SQLite documentation. https://www.sqlite.org/docs.html

Python sqlite3 module documentation. https://docs.python.org/3/library/sqlite3.html

pandas documentation: SQL queries. https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. Chapter 6 covers database interaction. https://wesmckinney.com/book/

Real Python: Data Management With Python, SQLite, and SQLAlchemy. https://realpython.com/python-sqlite-sqlalchemy/

## Cleanup

In [ ]:
# Optional: Remove database files created during this tutorial
import os

files_to_remove = ['college.db', 'bookshop.db', 'music.db']

# Uncomment to clean up:
# for f in files_to_remove:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"Removed {f}")